In [1]:
from pathlib import Path

import pandas as pd
import duckdb

path = Path(
    "/home/georgiusbenedikt09/MLOps_Credit_Risk/dataset/processed/train_base_depth2.parquet"
)
con = duckdb.connect(config={"threads": 2})

## Null Summary

In [2]:
# Read the parquet metadata
x = con.execute("SELECT * FROM parquet_metadata(?)", [str(path)]).df()
display(x.loc[:, ['path_in_schema', 'stats_null_count']])

,path_in_schema,stats_null_count
0,case_id,0
1,date_decision,0
2,MONTH,0
3,WEEK_NUM,0
4,target,0
...,...,...
30169,person_2_empls_economicalst_849M_distinct_coun...,24
30170,person_2_empls_employedfrom_796D_min,14002
30171,person_2_empls_employedfrom_796D_max,14002
30172,person_2_empls_employer_name_740M_distinct_cou...,24


In [3]:
# Compute the null counts and percentages for each feature in the parquet file
null_summary = con.execute(
    """
    SELECT
        path_in_schema AS feature,
        CAST(SUM(stats_null_count) AS BIGINT) AS null_count,
        ROUND(100.0 * SUM(stats_null_count) / SUM(row_group_num_rows), 5) AS null_pct
    FROM parquet_metadata(?)
    GROUP BY path_in_schema
    ORDER BY null_count DESC, feature
    """,
    [str(path)],
).df()

display(null_summary.head(40))

,feature,null_count,null_pct
0,person_1_applicant_childnum_185L,1526659,100.00000
1,person_1_applicant_housingtype_772L,1526659,100.00000
2,person_1_applicant_relationshiptoclient_415T,1526659,100.00000
3,person_1_applicant_relationshiptoclient_642T,1526659,100.00000
4,person_1_applicant_remitter_829L,1526659,100.00000
5,person_1_applicant_maritalst_703L,1526658,99.99993
6,static_0_clientscnt_136L,1526242,99.97269
7,static_0_lastrepayingdate_696D,1524282,99.84430
8,static_0_lastotherinc_902A,1523603,99.79982
9,static_0_lastotherlnsexpense_631A,1523601,99.79969


## Validate the input.

Check that `case_id` is unique and non-null, target contains only `0` and `1`, and every case has a `WEEK_NUM`.

In [4]:
# Check case_id uniqueness and is non-null
row_count = con.execute("SELECT COUNT(*) AS row_count FROM read_parquet(?)", [str(path)]).df().iloc[0, 0]
unique_case_ids = con.execute("SELECT COUNT(DISTINCT case_id) AS unique_case_ids FROM read_parquet(?)", [str(path)]).df().iloc[0, 0]
print(row_count == unique_case_ids)

True


In [5]:
# Check target only contains 0 and 1
target_values = con.execute("SELECT DISTINCT target FROM read_parquet(?)", [str(path)]).df()
display(target_values)

,target
0,1
1,0


In [6]:
# Check the null summary for the feature "WEEK_NUM"
display(null_summary.loc[null_summary["feature"] == "WEEK_NUM"])

,feature,null_count,null_pct
557,WEEK_NUM,0,0.0


## Standardize types

In [7]:
# Find date columns that is with string data type, rather than date data type.
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(path)],
).df()

names = schema["column_name"]

is_text = schema["column_type"].eq("VARCHAR")
is_date_name = names.str.endswith("D") | names.eq("date_decision")

string_dates = schema.loc[is_text & is_date_name, ["column_name", "column_type"]]
display(string_dates)

,column_name,column_type
1,date_decision,VARCHAR
49,static_0_datefirstoffer_1144D,VARCHAR
50,static_0_datelastinstal40dpd_247D,VARCHAR
51,static_0_datelastunpaid_3546854D,VARCHAR
57,static_0_dtlastpmtallstes_4499206D,VARCHAR
61,static_0_firstclxcampaign_1125D,VARCHAR
62,static_0_firstdatedue_489D,VARCHAR
71,static_0_lastactivateddate_801D,VARCHAR
72,static_0_lastapplicationdate_877D,VARCHAR
76,static_0_lastapprdate_640D,VARCHAR


In [8]:
# convert the string date columns to date data type
date_casts = ", ".join([f"CAST({name} AS DATE) AS {name}" for name in string_dates["column_name"]])

cleaned = con.sql(
    f"SELECT * REPLACE ({date_casts}) FROM read_parquet(?)",
    params=[str(path)],
)

preview = (
    cleaned.filter("date_decision >= DATE '2020-01-01'")
    .select("case_id", "date_decision")
    .limit(5)
    .df()
)

display(preview.info())

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   case_id        5 non-null      int64         
 1   date_decision  5 non-null      datetime64[us]
dtypes: datetime64[us](1), int64(1)
memory usage: 212.0 bytes


None

## Count infinites

In [9]:
# Check for infinite values in float columns
float_columns = [
    name for name, dtype in zip(cleaned.columns, cleaned.types)
    if str(dtype) in {"FLOAT", "DOUBLE"}
]

infinity_checks = ", ".join(
    f'COUNT(*) FILTER (WHERE isinf("{name}")) AS "{name}"'
    for name in float_columns
)

inf_counts = cleaned.aggregate(infinity_checks).df().iloc[0]
inf_columns = inf_counts[inf_counts > 0].sort_values(ascending=False)
display(inf_columns)

Series([], Name: 0, dtype: int64)

In [10]:
# Convert infinite values to nulls in float columns (if any)
if not inf_columns.empty:
    replacements = ", ".join(
        f'CASE WHEN isinf("{name}") THEN NULL ELSE "{name}" END AS "{name}"'
        for name in inf_columns.index
    )
    cleaned = cleaned.project(f"* REPLACE ({replacements})")

## Convert blank text to null

Count empty and whitespace-only strings, then replace them with SQL NULL in the affected text columns.

In [11]:
text_columns = [
    name for name, dtype in zip(cleaned.columns, cleaned.types)
    if str(dtype) == "VARCHAR"
]

# Build one blank-text count per column in a single DuckDB query.
blank_checks = []
for name in text_columns:
    column = '"' + name.replace('"', '""') + '"'
    blank_checks.append(rf"COUNT(*) FILTER (WHERE regexp_full_match({column}, '\s*'))")

[blank_counts] = cleaned.aggregate(", ".join(blank_checks)).fetchall()
placeholder_summary = pd.DataFrame({
    "feature": text_columns,
    "blank_count": blank_counts,
})

blank_fields = placeholder_summary.loc[placeholder_summary["blank_count"] > 0]
print(f"Found blanks in {len(blank_fields)} of {len(text_columns)} text columns.")
display(blank_fields.sort_values("blank_count", ascending=False))

Found blanks in 0 of 47 text columns.


,feature,blank_count


In [12]:
replacements = []
for name in blank_fields["feature"]:
    column = '"' + name.replace('"', '""') + '"'
    replacements.append(
        rf"CASE WHEN regexp_full_match({column}, '\s*') THEN NULL ELSE {column} END AS {column}"
    )

if replacements:
    cleaned = cleaned.project(f"* REPLACE ({', '.join(replacements)})")

## Validate and save the cleaned training table

Check the case IDs, labels, and dates, then verify that no infinities or blank text remain.

Only when everything passes will it write a parquet file on the cleaned dataset.

In [13]:
from tempfile import TemporaryDirectory

# COUNT(DISTINCT case_id) excludes nulls and counts duplicate IDs once.
validation = cleaned.aggregate("""
    COUNT(*),
    COUNT(DISTINCT case_id),
    COUNT(*) FILTER (WHERE target IS NULL OR target NOT IN (0, 1)),
    COUNT(*) FILTER (WHERE WEEK_NUM IS NULL),
    COUNT(*) FILTER (WHERE date_decision IS NULL)
""")

[(row_count, unique_case_ids, invalid_targets, missing_weeks, missing_dates)] = validation.fetchall()

assert row_count > 0, "The cleaned training table is empty."
assert unique_case_ids == row_count, "case_id must be unique and non-null."
assert invalid_targets == 0, f"Found {invalid_targets:,} missing or non-binary targets."
assert missing_weeks == 0, f"Found {missing_weeks:,} rows with a missing WEEK_NUM."
assert missing_dates == 0, f"Found {missing_dates:,} rows with a missing date_decision."

# Run the same checks again on the cleaned values before saving.
[remaining_infinities] = cleaned.aggregate(infinity_checks).fetchall()
infinite_fields = [name for name, count in zip(float_columns, remaining_infinities) if count]
assert not infinite_fields, f"Infinite values remain in: {', '.join(infinite_fields)}."

[remaining_blanks] = cleaned.aggregate(", ".join(blank_checks)).fetchall()
blank_text_fields = [name for name, count in zip(text_columns, remaining_blanks) if count]
assert not blank_text_fields, f"Blank text remains in: {', '.join(blank_text_fields)}."

cleaned_path = Path(path).with_name("train_base_cleaned.parquet")
with TemporaryDirectory(dir=cleaned_path.parent) as temp_dir:
    staged_path = Path(temp_dir) / cleaned_path.name
    cleaned.write_parquet(str(staged_path), row_group_size=32768)
    staged_path.replace(cleaned_path)

print(f"Validated and saved {row_count:,} cases to {cleaned_path.resolve()}")

Validated and saved 1,526,659 cases to /home/georgiusbenedikt09/MLOps_Credit_Risk/dataset/processed/train_base_cleaned.parquet
